In [2]:
import json
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup
import pandas as pd
import ollama
import requests
import os


a = 0
with open('lista-exames.json', 'r', encoding='utf-8') as arquivo:
    dados_carregados = json.load(arquivo)

prompt = """
Você é um especialista em deteção de fraturas, eu preciso que me diga se nesse laudo contém fraturas ou não.
EU PRECISO QUE VOCÊ RESPONDA APENAS COM 0 OU 1, RESPONDA 0 QUANDO NÃO TIVER FRATURA E RESPONDA 1 QUANDO TIVER FRATURA
NÃO ADICIONE MAIS NENHUM CARACTERE ALÉM DO 0 OU 1. QUANDO VOCÊ SE DEPARAR COM UM TEXTO CONTENDO "FIXADOR METÁLICO" ISSO 
TAMBÉM SIGNIFICA UMA FRATURA.
"""

unique = list()

ModuleNotFoundError: No module named 'ollama'

In [1]:
for i in range(0,10000):
    if dados_carregados[i]['studyinstanceuid'] not in unique:
        
        unique.append(dados_carregados[i]['studyinstanceuid'])
        img = dados_carregados[i]['url_base_pacs'].replace('wado','') + 'rs/studies/' + dados_carregados[i]['studyinstanceuid'] + '/instances'
        texto = dados_carregados[i]['laudo']
    
        
        try:
            root = ET.fromstring(texto)
            texto_laudo = root[9][2][3][-1][2].text
        
        except ET.ParseError:
            soup = BeautifulSoup(texto, "html.parser")
            section_laudo = soup.find("section", id="laudo")
            texto_laudo = section_laudo.get_text(separator=" ", strip=True)

        print(texto_laudo, f' - Instancia {i}')
    
        
        response = ollama.chat(
            model="mixtral",
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": texto_laudo},
            ],
            options={"temperature": 0, "num_predict": 10}
        )
        
        resultado = response["message"]["content"].strip()
    
    
    
        
        pasta_fratura = "fratura_2"
        pasta_nao_fratura = "nao_fratura_2"
        os.makedirs(pasta_fratura, exist_ok=True)
        os.makedirs(pasta_nao_fratura, exist_ok=True)
        
        print("Buscando metadados das instâncias do estudo...")
        response_meta = requests.get(url_instancias, headers=headers_metadados)
        
        print(response_meta.status_code)
        if response_meta.status_code == 200:
            
            lista_instancias = response_meta.json()
            print(f"Encontrada(s) {len(lista_instancias)} instância(s). Iniciando download...\n")
        
            for i, instancia in enumerate(lista_instancias):
                try:
                    url_base_instancia = instancia["00081190"]["Value"][0]
                except KeyError:
                    print(f"Instância {i+1} não possui a tag de URL (00081190). Pulando...")
                    continue
                
                url_download = f"{url_base_instancia}/rendered"
                
                print(f"[{i+1}/{len(lista_instancias)}] Baixando imagem...")
                
                response_img = requests.get(url_download, headers=headers_imagem)
        
                print(response_img.status_code)
                if response_img.status_code == 200:
                    print(a, resultado[0])
                    if resultado[0] == '1':
                        caminho_arquivo = os.path.join(pasta_fratura, f"instancia_{i}_{a}_{resultado[0]}_.png")
                    elif resultado[0] =='0':
                        caminho_arquivo = os.path.join(pasta_nao_fratura, f"instancia_{i}_{a}_{resultado[0]}_.png")
                    a+=1
                    
                    with open(caminho_arquivo, "wb") as f:
                        f.write(response_img.content)
                        
                    print(f"{caminho_arquivo}")
                else:
                    print(f"{response_img.status_code}")
        
        else:
            print(f"{response_meta.status_code}")
            print("Detalhes do erro:", response_meta.text)

NameError: name 'dados_carregados' is not defined

In [125]:
soup = BeautifulSoup(texto, "html.parser")

section_laudo = soup.find("section", id="laudo")

texto_laudo = section_laudo.get_text(separator=" ", strip=True)

print(texto_laudo)

LAUDO Número: 25701850 Data: 23/01/2026 10:35 Responsável: SILVIA RENATA CARVALHO - CRM/SC 7557 - RQE 7111 PÉS Esporão plantar e aquileu  bilateral . Osteófitos na extremidade distal dos 1º metatarsos . Redução do espaço MTT-falangeano no hálux esquerdo . JOELHOS Sinais de osteoartrose . Redução dos espaços fêmuro-tibiais mediais .


In [99]:
for elem in root.iter():
    if elem.text and elem.text.strip():
        resultado = elem.text
        print(f"<{elem.tag}> → {elem.text}")

<sopclass> → Basic SR
<modality> → SR
<charset> → ISO_IR 192
<first> → JAYME AUGUSTO HAACK
<last> → MENDONCA
<suffix> → Dr
<id> → 10583166
<first> → JOSE DARIO SEM
<last> → INTERNACAO
<date> → 1990-07-05
<sex> → M
<description> → Laudo do exame 22376155
<number> → 1
<description> → Laudo do exame 22376155
<number> → 1
<date> → 2025-04-23
<time> → 18:15:17
<uid> → 1.2.826.0.1.3680043.8.213.91
<name> → 99_CSR
<organization> → The Cyclops Group
<date> → 2025-04-23
<time> → 18:15:17
<date> → 2025-04-23
<time> → 18:15:17
<codevalue> → CRM/SC 24566
<codingschemedesignator> → 99_CONSTRABALHO
<value> → CR
<designator> → 99_CSR
<meaning> → Laudo do exame nº 22376155
<relationship> → CONTAINS
<value> → CSR_REQUEST
<designator> → 99_CSR
<meaning> → Requisição
<value> → Radiologia Computadorizada requisição nº 2025098929-12
<relationship> → CONTAINS
<value> → CSR_FINDINGS
<designator> → 99_CSR
<meaning> → Achados
<value> → <p>Controle radiológico.</p><p>Fixador metálico externo.</p>


In [47]:
print(resultado)

<p>Estruturas ósseas íntegras.<br>Espaços articulares preservados.<br>Partes moles sem anormalidades.</p>


In [30]:
print('1') if 'Fratura' in dados_carregados[i]['laudo'] else '0'

'0'

In [145]:
texto_laudo

'<p>Estrutura e densidade óssea de aspectos normais.&nbsp;</p><p>Superfícies articulares íntegras, com espaços preservados.&nbsp;</p><p>Não há sinal de fratura.&nbsp;</p><p>&nbsp;</p>'